# REPORT1 SXO experiment pitch assets

This notebook builds the chart exports used in `docs/REPORT1_sxo_experiment_pitch.md`.
It prefers live BigQuery data from the same warehouse tables used in `REPORT1.md`, and falls back to report snapshot values if credentials are unavailable.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import date
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.cloud import bigquery

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import lifeline_theme
import lla_data.config as config
from lla_data.bq import get_client, load_sql_template, run_query

pd.set_option('display.max_colwidth', 140)
lifeline_theme.inject_fonts()

ANALYSIS_START_DATE = date(2025, 12, 14)
ANALYSIS_END_DATE = date(2026, 3, 12)
PAGE_PREFIXES = [
    '/get-help/support-toolkit',
    '/get-help/national-services',
    '/get-help/hear-from-others',
]
FOCUS_PAGES = {
    '/get-help/support-toolkit/techniques-and-guides/self-care-for-mental-health-and-wellbeing/': {
        'label': 'Self-care',
        'target_ctr': 0.0022,
    },
    '/get-help/support-toolkit/techniques-and-guides/finding-the-right-therapist/': {
        'label': 'Finding the right therapist',
        'target_ctr': 0.0012,
    },
    '/get-help/support-toolkit/techniques-and-guides/finding-relief-through-grounding-techniques/': {
        'label': 'Grounding techniques',
        'target_ctr': 0.0085,
    },
}
EXPORT_DIR = REPO_ROOT / 'docs' / 'assets' / 'report1_experiment'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

client = None
try:
    client = get_client()
except Exception as exc:
    print(f'BigQuery client unavailable, using report snapshot fallback: {exc}')


In [ ]:
report_segment_snapshot = pd.DataFrame(
    [
        {'content_segment': 'Techniques and guides', 'impressions': 4_800_000, 'clicks': 18_200, 'ctr': 0.0038, 'organic_sessions': 26_400},
        {'content_segment': 'National services', 'impressions': 4_240_000, 'clicks': 42_100, 'ctr': 0.0099, 'organic_sessions': 53_200},
        {'content_segment': 'Tools and apps', 'impressions': 1_540_000, 'clicks': 15_800, 'ctr': 0.0103, 'organic_sessions': 19_700},
        {'content_segment': 'Topics', 'impressions': 1_920_000, 'clicks': 8_900, 'ctr': 0.0046, 'organic_sessions': 16_500},
        {'content_segment': 'Hear from others', 'impressions': 980_000, 'clicks': 2_500, 'ctr': 0.0026, 'organic_sessions': 3_800},
        {'content_segment': 'Other self-led support', 'impressions': 720_000, 'clicks': 0, 'ctr': 0.0, 'organic_sessions': 3_500},
    ]
)

report_focus_snapshot = pd.DataFrame(
    [
        {
            'page_path': '/get-help/support-toolkit/techniques-and-guides/self-care-for-mental-health-and-wellbeing/',
            'impressions': 748_600,
            'clicks': 1_100,
            'ctr': 0.0015,
            'avg_position': 6.5,
            'organic_sessions': 1_450,
            'engaged_sessions': 1_000,
        },
        {
            'page_path': '/get-help/support-toolkit/techniques-and-guides/finding-the-right-therapist/',
            'impressions': 492_600,
            'clicks': 368,
            'ctr': 0.0007,
            'avg_position': 4.7,
            'organic_sessions': 520,
            'engaged_sessions': 360,
        },
        {
            'page_path': '/get-help/support-toolkit/techniques-and-guides/finding-relief-through-grounding-techniques/',
            'impressions': 209_400,
            'clicks': 1_400,
            'ctr': 0.0066,
            'avg_position': 7.3,
            'organic_sessions': 1_950,
            'engaged_sessions': 1_420,
        },
    ]
)

report_head_terms_snapshot = pd.DataFrame(
    [
        {
            'page_path': '/get-help/support-toolkit/techniques-and-guides/self-care-for-mental-health-and-wellbeing/',
            'query_cluster': 'Self-care ideas cluster',
            'impressions': 474_100,
            'clicks': 711,
            'ctr': 0.0015,
            'avg_position': 6.5,
        },
        {
            'page_path': '/get-help/support-toolkit/techniques-and-guides/finding-the-right-therapist/',
            'query_cluster': 'Therapist choice cluster',
            'impressions': 397_300,
            'clicks': 278,
            'ctr': 0.0007,
            'avg_position': 4.7,
        },
        {
            'page_path': '/get-help/support-toolkit/techniques-and-guides/finding-relief-through-grounding-techniques/',
            'query_cluster': 'Grounding techniques cluster',
            'impressions': 150_000,
            'clicks': 990,
            'ctr': 0.0066,
            'avg_position': 7.3,
        },
    ]
)

def query_or_fallback(sql_path: str, params: list[bigquery.ScalarQueryParameter], fallback_df: pd.DataFrame) -> pd.DataFrame:
    if client is None:
        return fallback_df.copy()
    try:
        sql = load_sql_template(
            sql_path,
            project_id=config.PROJECT_ID,
            searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
        )
        return run_query(client, sql, params=params)
    except Exception as exc:
        print(f'Falling back for {sql_path}: {exc}')
        return fallback_df.copy()


In [ ]:
segment_params = [
    bigquery.ScalarQueryParameter('start_date', 'DATE', ANALYSIS_START_DATE),
    bigquery.ScalarQueryParameter('end_date', 'DATE', ANALYSIS_END_DATE),
    bigquery.ArrayQueryParameter('page_prefixes', 'STRING', PAGE_PREFIXES),
]
page_params = [
    bigquery.ScalarQueryParameter('start_date', 'DATE', ANALYSIS_START_DATE),
    bigquery.ScalarQueryParameter('end_date', 'DATE', ANALYSIS_END_DATE),
    bigquery.ArrayQueryParameter('pages', 'STRING', list(FOCUS_PAGES)),
]
head_term_params = [
    bigquery.ScalarQueryParameter('start_date', 'DATE', ANALYSIS_START_DATE),
    bigquery.ScalarQueryParameter('end_date', 'DATE', ANALYSIS_END_DATE),
]

df_segments = query_or_fallback('sql/notebooks/seo_sxo_pitch_cohort_segments.sql', segment_params, report_segment_snapshot)
df_pages = query_or_fallback('sql/notebooks/seo_sxo_pitch_focus_pages.sql', page_params, report_focus_snapshot)
df_head_terms = query_or_fallback('sql/notebooks/seo_sxo_pitch_head_terms.sql', head_term_params, report_head_terms_snapshot)

df_pages['page_label'] = df_pages['page_path'].map(lambda value: FOCUS_PAGES[value]['label'])
df_pages['target_ctr'] = df_pages['page_path'].map(lambda value: FOCUS_PAGES[value]['target_ctr'])
df_head_terms['page_label'] = df_head_terms['page_path'].map(lambda value: FOCUS_PAGES[value]['label'])

df_segments


In [ ]:
segment_plot = df_segments.copy()
segment_plot['impressions_m'] = segment_plot['impressions'] / 1_000_000
segment_plot['clicks_k'] = segment_plot['clicks'] / 1_000
segment_plot['organic_sessions_k'] = segment_plot['organic_sessions'] / 1_000
segment_plot = segment_plot.sort_values('impressions', ascending=False)

fig_segments = go.Figure()
fig_segments.add_bar(
    x=segment_plot['content_segment'],
    y=segment_plot['impressions_m'],
    name='Impressions (M)',
)
fig_segments.add_bar(
    x=segment_plot['content_segment'],
    y=segment_plot['clicks_k'],
    name='Clicks (K)',
)
fig_segments.add_bar(
    x=segment_plot['content_segment'],
    y=segment_plot['organic_sessions_k'],
    name='Organic sessions (K)',
)
fig_segments.add_scatter(
    x=segment_plot['content_segment'],
    y=segment_plot['ctr'] * 100,
    name='CTR %',
    mode='lines+markers',
    yaxis='y2',
)
fig_segments.update_layout(
    template='lifeline',
    title='Self-led support cohort: scale is high, CTR is uneven',
    barmode='group',
    xaxis_title='',
    yaxis_title='Volume',
    yaxis2=dict(title='CTR %', overlaying='y', side='right', showgrid=False),
    legend_title='',
    width=1600,
    height=900,
)
lifeline_theme.add_lifeline_logo(fig_segments)
fig_segments.write_image(EXPORT_DIR / 'cohort-context-by-segment.png', scale=2)
fig_segments.show()


In [ ]:
page_plot = df_pages.copy().sort_values('impressions', ascending=False)
page_plot['impressions_k'] = page_plot['impressions'] / 1_000
page_plot['clicks'] = page_plot['clicks'].astype(float)
page_plot['ctr_pct'] = page_plot['ctr'] * 100

fig_pages = make_subplots(rows=1, cols=2, specs=[[{'type': 'bar'}, {'type': 'bar'}]], subplot_titles=('Search opportunity', 'Current efficiency'))
fig_pages.add_bar(x=page_plot['page_label'], y=page_plot['impressions_k'], name='Impressions (K)', row=1, col=1)
fig_pages.add_bar(x=page_plot['page_label'], y=page_plot['clicks'], name='Clicks', row=1, col=1)
fig_pages.add_bar(x=page_plot['page_label'], y=page_plot['ctr_pct'], name='CTR %', row=1, col=2)
fig_pages.add_bar(x=page_plot['page_label'], y=page_plot['avg_position'], name='Avg position', row=1, col=2)
fig_pages.update_layout(
    template='lifeline',
    title='Focus pages: strong visibility, mixed click-through performance',
    barmode='group',
    legend_title='',
    width=1800,
    height=900,
)
fig_pages.update_yaxes(title_text='Volume', row=1, col=1)
fig_pages.update_yaxes(title_text='CTR % / avg position', row=1, col=2)
lifeline_theme.add_lifeline_logo(fig_pages)
fig_pages.write_image(EXPORT_DIR / 'focus-pages-comparison.png', scale=2)
fig_pages.show()


In [ ]:
target_plot = df_pages[['page_label', 'ctr', 'target_ctr']].copy().sort_values('target_ctr', ascending=False)
target_plot['baseline_pct'] = target_plot['ctr'] * 100
target_plot['target_pct'] = target_plot['target_ctr'] * 100

fig_targets = go.Figure()
for _, row in target_plot.iterrows():
    fig_targets.add_trace(
        go.Scatter(
            x=[row['baseline_pct'], row['target_pct']],
            y=[row['page_label'], row['page_label']],
            mode='lines+markers',
            marker=dict(size=14),
            line=dict(width=6),
            name=row['page_label'],
            showlegend=False,
            hovertemplate='CTR: %{x:.2f}%<extra></extra>',
        )
    )
fig_targets.update_layout(
    template='lifeline',
    title='CTR targets for the first SXO experiment',
    xaxis_title='CTR %',
    yaxis_title='',
    width=1600,
    height=850,
)
lifeline_theme.add_lifeline_logo(fig_targets)
fig_targets.write_image(EXPORT_DIR / 'ctr-targets.png', scale=2)
fig_targets.show()

df_head_terms[['page_label', 'query_cluster', 'impressions', 'clicks', 'ctr', 'avg_position']]
